---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-38: Hands-on Examples of <b>Vectorized and Vectorless RAG</b></h1>

# Learning agenda of this notebook

> **The idea of this lecture:** Lec-30 to 37 gave us every building block: models, prompts, chains, loaders, splitters, vector stores and retrievers. This lecture puts them together into **Retrieval-Augmented Generation (RAG)**, the most common way to let an LLM answer questions about *your* documents. We look at the types of RAG used today, build a complete pipeline step by step, and end with a RAG that uses **no vectors at all**.

1. **What is RAG?**
2. **Naive RAG, and Where it Breaks**
3. **RAG Paradigms and Types**
4. **Advanced RAG Techniques**
5. **A Complete RAG Pipeline, Step by Step**
    - Example 1: Ask questions about a research paper
6. **Two Ready-to-Use RAG Apps**
    - Example 2: Ask questions about a YouTube lecture
    - Example 3: Chat with any PDF
7. **Vectorless RAG**
    - Example 4: An LLM reads the table of contents instead of searching vectors
8. **Summary**

> **Note:** all models are free: `all-MiniLM-L6-v2` (embeddings) and `ms-marco-MiniLM-L6-v2` (reranking) run locally, and `openai/gpt-oss-120b` runs on Groq.

**Further reading**
- [Retrieval-Augmented Generation for Large Language Models: A Survey (Gao et al., 2023-2024)](https://arxiv.org/abs/2312.10997)
- [Enhancing RAG: A Study of Best Practices](https://github.com/arifpucit/Generative-and-Agentic-AI/blob/main/Research%20Articles/39-Enhancing_RAG_A_Study_of_Best_Practices.pdf)
- [Seven Failure Points When Engineering a RAG System (Barnett et al., CAIN 2024)](https://arxiv.org/abs/2401.05856)
- [Agentic Retrieval-Augmented Generation: A Survey on Agentic RAG (Singh et al., 2025)](https://arxiv.org/abs/2501.09136)
- [Multi-RAG: A Multimodal RAG System for Adaptive Video Understanding (Jun 2025)](https://arxiv.org/pdf/2505.23990v2)

# <span style='background :lightgreen' >1. What is RAG?</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">RAG lets a language model answer from <b>your documents</b>: first find the relevant passages, then give them to the model together with the question.</div></h3>

- **A real-world analogy:**
    - A plain LLM is a student in a **closed-book** exam: it can only use what it memorised during training.
    - A RAG system is a student in an **open-book** exam: it looks up the right pages first, then writes the answer.
- **The three letters:**
    - **Retrieval:** find the passages that are relevant to the question (with vectors, keywords, or both);
    - **Augmented:** add those passages to the prompt, next to the question;
    - **Generation:** the LLM writes the answer from that prompt.
- **Why RAG?**
    - **Up-to-date and private knowledge:** the model's training stopped at some date and never saw your files.
    - **Fewer made-up answers:** the model is told to answer only from the passages, and to say "I don't know" otherwise.
    - **Sources:** every answer can cite the page or time it came from.
    - **Cheap to update:** add a document to the index; no fine-tuning (Lec-22 to 26) is needed.

How a document becomes searchable data:

In [1]:
from IPython.display import HTML                 # to show an HTML video player in the notebook

HTML("""
<div style="text-align: center;">
  <video id="myvideo" width="700" height="400" controls style="display: inline-block;">
    <source src="../images/How-data-is-stored.mp4" type="video/mp4">
  </video>
</div>
<script>
  document.getElementById('myvideo').playbackRate = 0.5;   // play at half speed
</script>
""")

The whole RAG pipeline in one picture, with the real values of Example 1:

<div style="text-align:center;">
    <img src="../images/langchain-new-31.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

- **Indexing** happens once: load, split, embed and store the documents (Lec-34 to 36).
- **Querying** happens for every question: retrieve the best chunks (Lec-37), put them in a prompt (Lec-31), and let the model answer (Lec-30). With `|` the three steps become one chain (Lec-33).

# <span style='background :lightgreen' >2. Naive RAG, and Where it Breaks</span>

The simplest version, **naive RAG**, does exactly the steps above with default settings: fixed-size chunks, vector search for the top `k` chunks, and a plain prompt. It is a good start, but a study of real RAG systems (Barnett et al., 2024) found **seven points where it fails**. Each one has a fix that you have already seen:

| # | Failure point | What goes wrong | Fix | Where |
|---|---|---|---|---|
| 1 | Missing content | the answer is not in the documents at all | say "I don't know" instead of guessing | Ex 1 |
| 2 | Missed top-ranked documents | the right chunk exists but is not in the top `k` | hybrid search, reranking, multi-query | Lec-37, Ex 1 |
| 3 | Not in context | too many chunks; the right one is cut when the prompt is built | fewer, better chunks after reranking | Lec-37 |
| 4 | Not extracted | the answer is in the prompt, but the model misses it (noise, contradictions) | rerank, compress, clear instructions | Ex 1 |
| 5 | Wrong format | the user asked for a table or list and got prose | structured output, format instructions | Lec-30, 33 |
| 6 | Incorrect specificity | the answer is too general or too detailed | better prompts, query rewriting | Lec-31, 37 |
| 7 | Incomplete | only part of the answer was found | decomposition, more chunks, agentic RAG | Sec. 3, 4 |

Two more problems come from **loading and splitting**, before retrieval even starts: tables read as jumbled numbers (Lec-35, Example 7) and chunks too long for the embedding model (Lec-35, Example 1).

# <span style='background :lightgreen' >3. RAG Paradigms and Types</span>

Since the first RAG paper (Lewis et al., 2020), many variants have appeared. A survey by Gao et al. groups them into three **paradigms** (naive, advanced, modular); newer types change *when* and *how* retrieval happens, or *how the knowledge is organised*:

<div style="text-align:center;">
    <img src="../images/langchain-new-32.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

The main difference between the types is the **control flow**: a straight line, a route, a correction step, or a loop:

<div style="text-align:center;">
    <img src="../images/langchain-new-33.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

| RAG type | Key idea | When to use | Source |
|---|---|---|---|
| **Naive RAG** | index → retrieve top-k → prompt → generate | prototypes, small and clean document sets | Lewis et al., NeurIPS 2020 |
| **Advanced RAG** | adds steps **before** retrieval (better chunks, query rewriting) and **after** it (reranking, compression) | most production systems | Gao et al., survey 2023 |
| **Modular RAG** | the pipeline is built from swappable modules: routers, several retrievers, loops | systems that combine many sources and strategies | Gao et al., survey 2023 |
| **Self-RAG** | a specially **trained** model decides when to retrieve and grades its passages and its own answer with "reflection tokens" | when you can use or train such a model | Asai et al., ICLR 2024 |
| **Corrective RAG (CRAG)** | a light **evaluator** grades the retrieved chunks; if they are poor, the question is rewritten and the **web** is searched | when the documents may not contain the answer | Yan et al., 2024 |
| **Adaptive-RAG** | a small classifier routes each question: **no** retrieval, **one** retrieval, or **several** steps | mixed traffic of easy and hard questions | Jeong et al., NAACL 2024 |
| **Speculative RAG** | a **small** model writes several drafts, each from a different subset of the chunks; a **large** model checks them and picks the best | when both speed and accuracy matter | Wang et al., ICLR 2025 |
| **GraphRAG** | builds a **knowledge graph** of entities and summaries of their groups; answers questions about a whole collection | "What are the main themes in these 1,000 reports?" | Edge et al. (Microsoft), 2024 |
| **Agentic RAG** | an **agent** plans, chooses tools (retrievers, web, SQL), checks results and repeats | multi-step questions, many sources | Singh et al., survey 2025 |
| **Vectorless RAG** | no embeddings: an LLM reads the document's **table of contents** and picks the sections to read | long, well-structured documents (reports, contracts, manuals, papers) | PageIndex, 2025 (Example 4) |

# <span style='background :lightgreen' >4. Advanced RAG Techniques</span>

Advanced RAG improves every step of the pipeline. Most of these techniques were examples in Lec-34 to 37:

<div style="text-align:center;">
    <img src="../images/langchain-new-34.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

### (i) Techniques before retrieval

| Technique | What it does | Problem it solves | Example |
|---|---|---|---|
| **Smart chunking** | chunks sized in the embedding model's tokens, split at paragraphs and headings | chunks cut in the middle of an idea, or silently truncated | Lec-35, Examples 1, 5, 6 |
| **Metadata** | add page, source, date, type to every chunk | cannot cite or filter by source | "only Q3 2024 financial reports" (Lec-36, Ex 3) |
| **Small-to-big** | search small chunks, but give the model the larger section around them | small chunks lack context for the answer | find "Python uses garbage collection", return the whole memory-management section |
| **Contextual retrieval** | before embedding, add a sentence that says where the chunk comes from in the document | a chunk such as "Revenue grew 3%" does not say *whose* revenue | Anthropic, 2024: with BM25 and reranking, failed retrievals fell from 5.7% to 1.9% |
| **Multi-query** | the LLM writes several versions of the question | the user's words differ from the document's words | Lec-37, Ex 5 |
| **HyDE** | the LLM writes an imagined answer; that answer is used as the search query | questions and answers look different as vectors | Gao et al., 2022 |
| **Decomposition** | split a complex question into simpler sub-questions | one search cannot answer "compare A and B" | "How does Python manage memory?" + "How does Java manage memory?" |
| **Routing** | send the question to the right source or tool | one index for everything | SQL for numbers, vector store for policies |

### (ii) Techniques during and after retrieval

| Technique | What it does | Problem it solves | Example |
|---|---|---|---|
| **Hybrid search** | BM25 + vectors, merged with rank fusion | vectors miss codes and names; BM25 misses meanings | Lec-37, Ex 7; Example 1 below |
| **Reranking** | a cross-encoder re-scores the candidates | the first ranking is fast but rough | Lec-37, Ex 6; Example 1 below |
| **Score threshold** | drop chunks below a relevance score | weak chunks add noise, or the answer is not in the documents | Lec-37, Ex 2 |
| **Compression** | keep only the sentences that matter | long prompts cost more and distract the model | 3 chunks of 2,000 tokens → 600 tokens of relevant sentences |

### (iii) Generation and evaluation

| Technique | What it does |
|---|---|
| **Grounded prompt** | "Answer only from the context; if it is not there, say *I don't know*" (Example 1) |
| **Citations** | the context is labelled with pages or times, and the model cites them: `[p.7]`, `[01:24]` (Examples 1 to 3) |
| **Evaluation** | test questions with known answers, and metrics such as **faithfulness** (is every claim in the context?), **answer relevance**, and **context precision / recall** (were the right chunks retrieved?). Tools: RAGAS, LangSmith |

### (iv) Which RAG should I use?
```
Start with Naive RAG and a set of test questions
    │
    ├─ wrong chunks retrieved?           → Advanced RAG: hybrid search + reranking (Example 1)
    ├─ answer often not in the documents? → Corrective RAG: grade, then search the web
    ├─ many easy questions, some hard?   → Adaptive-RAG: route by difficulty
    ├─ questions about a whole collection ("main themes")? → GraphRAG
    ├─ long, well-structured documents?  → Vectorless RAG (Example 4)
    └─ multi-step tasks, many sources?   → Agentic RAG (Lec-39 to 53)
```

# <span style='background :lightgreen' >5. A Complete RAG Pipeline, Step by Step</span>

## <span style='color:#7e22ce'>Example 1: Ask questions about a research paper</span>

We build an **advanced RAG** pipeline for `attention.pdf`, the paper we used in Lec-34 to 37, one step per cell. Each step is one component from an earlier lecture.

### Setup: the models

In [2]:
import os, re                                                       # environment variables, regular expressions
import transformers                                                 # the library under sentence-transformers
from dotenv import load_dotenv                                      # load the API keys from ../keys/.env
from transformers import AutoTokenizer                              # to size chunks in MiniLM tokens (Lec-35)
from langchain_openai import ChatOpenAI                             # works with Groq's OpenAI-compatible API (Lec-30)
from langchain_huggingface import HuggingFaceEmbeddings             # the free local embedding model (Lec-30)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder   # the free local reranker (Lec-37)

transformers.logging.set_verbosity_error()                          # hide library warnings
transformers.logging.disable_progress_bar()                         # hide the "Loading weights" progress bar
load_dotenv("../keys/.env", override=True)
print("GROQ_API_KEY", "set" if os.getenv("GROQ_API_KEY") else "MISSING")   # never print any part of a key

GROQ_MODEL = ChatOpenAI(model="openai/gpt-oss-120b", base_url="https://api.groq.com/openai/v1",
                        api_key=os.getenv("GROQ_API_KEY"), max_retries=5)   # free; writes the answers
EMBED_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBED_NAME)            # 384 numbers per text, local
tokenizer = AutoTokenizer.from_pretrained(EMBED_NAME)                # the same tokenizer, to count tokens
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L6-v2")   # local reranker

def words(text: str) -> list[str]:
    """Lower-case words of letters and digits, for BM25 (Lec-37, Example 7)."""
    return re.findall(r"\w+", text.lower())

GROQ_API_KEY set


## <span style='background :lightblue' >Indexing: once, before any question</span>
### Step 1: Load the document (Lec-34)

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("../data/attention.pdf").load()          # one Document per page
print(len(pages), "pages; metadata of page 3:", {k: pages[2].metadata[k] for k in ["source", "page_label"]})

15 pages; metadata of page 3: {'source': '../data/attention.pdf', 'page_label': '3'}


### Step 2: Split into chunks (Lec-35)
At most 200 MiniLM tokens per chunk, so that every chunk fits the embedding model (256 tokens), with a small overlap. Every chunk keeps the page number of its page.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer, chunk_size=200, chunk_overlap=20)               # measured in MiniLM tokens
chunks = splitter.split_documents(pages)                       # Documents in, Documents out
print(len(pages), "pages ->", len(chunks), "chunks")
print("chunk 10, from page", chunks[10].metadata["page_label"], ":", chunks[10].page_content[:120], "...")

15 pages -> 60 chunks
chunk 10, from page 3 : itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce  ...


### Step 3: Embed and store (Lec-36)

In [5]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    chunks, embeddings,                                   # embed all chunks with MiniLM
    collection_name="attention_paper",
    collection_metadata={"hnsw:space": "cosine"},         # cosine distance (Lec-36, Example 2)
)
print("stored:", len(vector_store.get()["ids"]), "chunks with their vectors")

stored: 60 chunks with their vectors


## <span style='background :lightblue' >Querying: for every question</span>
### Step 4: Retrieve: hybrid search, then reranking (Lec-37)
- **Hybrid search:** BM25 and vector search each return 10 candidates; Reciprocal Rank Fusion merges them.
- **Reranking:** the cross-encoder reads the question with each candidate and keeps the best 4.

We compare it with plain vector search on the question that failed in Lec-36:

In [6]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})                 # meaning
bm25_retriever = BM25Retriever.from_documents(chunks, k=10, preprocess_func=words)    # exact words
hybrid = EnsembleRetriever(retrievers=[bm25_retriever, vector_retriever], weights=[0.5, 0.5])

retriever = ContextualCompressionRetriever(
    base_retriever=hybrid,                                                  # up to 20 candidates
    base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=4),    # keep the best 4
)

question = "Who are the authors of the paper?"
plain = vector_store.as_retriever(search_kwargs={"k": 4}).invoke(question)
best = retriever.invoke(question)
print("plain vector search :", ["p." + d.metadata["page_label"] for d in plain])
print("hybrid + reranking  :", ["p." + d.metadata["page_label"] for d in best])
print("\nbest chunk:", best[0].page_content[:300].replace("\n", " "), "...")

plain vector search : ['p.11', 'p.12', 'p.12', 'p.1']
hybrid + reranking  : ['p.1', 'p.12', 'p.12', 'p.11']

best chunk: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Par ...


Plain vector search puts the reference list (pages 11 and 12) first; hybrid search with reranking puts **page 1**, with the author names, first.

### Step 5: Augment: build the prompt (Lec-31)
The chunks become one **context** string, each labelled with its page, so that the model can cite it. The instructions tell the model to use only the context, to cite pages, and to say "I don't know" when the answer is missing.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

def format_docs(docs) -> str:
    """Join the chunks into one string, each one labelled with its page: [p.7] ..."""
    return "\n\n".join(f"[p.{d.metadata['page_label']}] {d.page_content}" for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about a document. Use ONLY the context below. "
               "After each fact, cite its page like [p.7]. "
               "If the answer is not in the context, say: I don't know."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

filled = prompt.invoke({"context": format_docs(best), "question": question})   # the messages the model will get
print(filled.messages[1].content[:600], "...")

Context:
[p.1] Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gm ...


### Step 6: Generate the answer (Lec-30)

In [8]:
from langchain_core.output_parsers import StrOutputParser

answer = (prompt | GROQ_MODEL | StrOutputParser()).invoke({"context": format_docs(best), "question": question})
print(answer)

The paper lists the following authors:

- Ashish Vaswani  
- Noam Shazeer  
- Niki Parmar  
- Jakob Uszkoreit  
- Llion Jones  
- Aidan N. Gomez  
- Łukasz Kaiser  
- Illia Polosukhin  

[p.1]


### Step 7: All steps as one chain (Lec-33)
Steps 4 to 6 become one LCEL chain. It returns the answer **and** the chunks it used, so that the app can show its sources:
- `RunnableParallel` runs the retriever and passes the question through, giving `{"docs": ..., "question": ...}`;
- `.assign(context=...)` adds the formatted context;
- `.assign(answer=...)` adds the answer from `prompt | model | parser`.

In [9]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

rag_chain = (
    RunnableParallel(docs=retriever, question=RunnablePassthrough())          # {"docs": [...], "question": "..."}
    | RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))    # + "context": one string
    | RunnablePassthrough.assign(answer=prompt | GROQ_MODEL | StrOutputParser())   # + "answer"
)

for q in ["How many attention heads does the base model use?",
          "What hardware was used to train the models, and for how long?",
          "What is the capital of France?"]:                                   # not in the paper
    result = rag_chain.invoke(q)
    pages_used = sorted({d.metadata["page_label"] for d in result["docs"]}, key=int)
    print("Q:", q)
    print("A:", result["answer"])
    print("   chunks from pages:", pages_used, "\n")

Q: How many attention heads does the base model use?
A: The base Transformer model uses eight parallel attention heads (h = 8) [p.5].
   chunks from pages: ['5', '7', '9'] 

Q: What hardware was used to train the models, and for how long?
A: The models were trained on a single machine equipped with **8 NVIDIA P100 GPUs**【p.7】.  

- **Base models** were trained for **100 000 steps**, which took about **12 hours**【p.7】.  
- **Big models** were trained for **300 000 steps**, which took roughly **3.5 days**【p.7】.
   chunks from pages: ['7', '8'] 

Q: What is the capital of France?
A: I don't know.
   chunks from pages: ['1', '3', '4', '14'] 



The first two answers cite their pages. For the last question, the retriever still returned 4 chunks (it always returns something), but the model found no answer in them and said **"I don't know"** instead of using its own knowledge: this is what the grounded prompt is for.

# <span style='background :lightgreen' >6. Two Ready-to-Use RAG Apps</span>

## <span style='color:#7e22ce'>Example 2: Ask questions about a YouTube lecture</span>

A video becomes searchable through its **transcript**. The `youtube-transcript-api` package downloads the captions as small pieces of text, each with its **start time**. We group them into 30-second chunks and keep the time in the metadata, so that every answer can point to the moment in the video.

Video: [Linux shell commands, introduction](https://www.youtube.com/watch?v=ndl79-VDl50) (about 4 minutes).

> **Note:** YouTube sometimes blocks requests (for example from cloud machines). The code then loads a saved copy of the same transcript from `../data/lec38_youtube_transcript.json`.

In [10]:
import json                                                      # to read the saved copy of the transcript
from langchain_core.documents import Document

VIDEO_ID = "ndl79-VDl50"                                         # the part after "watch?v=" in the URL
try:
    from youtube_transcript_api import YouTubeTranscriptApi
    fetched = YouTubeTranscriptApi().fetch(VIDEO_ID, languages=["en"])            # download the English captions
    snippets = [{"text": s.text, "start": s.start, "duration": s.duration} for s in fetched]
    print("transcript downloaded from YouTube")
except Exception as error:                                       # blocked, offline, or captions turned off
    with open("../data/lec38_youtube_transcript.json", encoding="utf-8") as f:
        snippets = json.load(f)["snippets"]
    print("YouTube not reachable (", type(error).__name__, "): using the saved transcript")

def mmss(seconds: float) -> str:
    """78.5 -> '01:18'"""
    return f"{int(seconds) // 60:02d}:{int(seconds) % 60:02d}"

video_chunks, group = [], []
for s in snippets:                                               # group the short pieces into ~30-second chunks
    group.append(s)
    if s["start"] + s["duration"] - group[0]["start"] >= 30 or s is snippets[-1]:
        video_chunks.append(Document(" ".join(g["text"] for g in group),
                                     metadata={"time": mmss(group[0]["start"])}))
        group = []

print(len(snippets), "caption pieces ->", len(video_chunks), "chunks starting at", [d.metadata["time"] for d in video_chunks])
print(video_chunks[0].page_content[:150], "...")

transcript downloaded from YouTube
78 caption pieces -> 9 chunks starting at ['00:00', '00:28', '00:55', '01:24', '01:53', '02:22', '02:52', '03:22', '03:50']
bismillah ar-rahman rahim assalamu alaikum dear students I welcome you to this series of lectures for linux shell commands these are long awaited by m ...


Now the same pipeline as Example 1, with times instead of pages, and a loop: type a question, read the answer, press **Enter** on an empty line to stop.

In [11]:
video_store = Chroma.from_documents(video_chunks, embeddings, collection_name="youtube_lecture",
                                    collection_metadata={"hnsw:space": "cosine"})
video_retriever = ContextualCompressionRetriever(
    base_retriever=EnsembleRetriever(retrievers=[
        BM25Retriever.from_documents(video_chunks, k=5, preprocess_func=words),
        video_store.as_retriever(search_kwargs={"k": 5})]),
    base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=3))       # the best 3 pieces of the video

video_prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about a video lecture. Use ONLY the transcript parts below. "
               "Cite the time of each part you use, like [01:30]. If the answer is not there, say: I don't know."),
    ("human", "Transcript parts:\n{context}\n\nQuestion: {question}"),
])
video_chain = (
    RunnableParallel(docs=video_retriever, question=RunnablePassthrough())
    | RunnablePassthrough.assign(context=lambda x: "\n\n".join(f"[{d.metadata['time']}] {d.page_content}" for d in x["docs"]))
    | RunnablePassthrough.assign(answer=video_prompt | GROQ_MODEL | StrOutputParser())
)

while True:                                                        # ask until the user presses Enter on an empty line
    question = input("Ask about the video (Enter to stop): ")
    if not question.strip():
        break
    result = video_chain.invoke(question)
    print("Answer:", result["answer"])
    print("   from the video at:", [d.metadata["time"] for d in result["docs"]], "\n")
print("Bye!")

Ask about the video (Enter to stop): Which inter-process communication methods will be covered?
Answer: The lecture will cover the following inter‑process communication methods:

- Signals  
- Pipes (including FIFO/named pipes)  
- Message queues  
- Shared memory  

These methods are listed in the transcript segment that says “using signals pipes … message queues and shared memory” [01:53].
   from the video at: ['01:24', '01:53', '00:55'] 

Ask about the video (Enter to stop): Which tools are used to understand system calls like fork, wait and exit?
Answer: The lecture says you can use **GCC and GDB** to explore system calls such as fork, wait and exit【03:22】.
   from the video at: ['03:22', '02:52', '00:28'] 

Ask about the video (Enter to stop): Who developed the Linux kernel?
Answer: I don't know.
   from the video at: ['01:24', '00:00', '00:28'] 

Ask about the video (Enter to stop): 
Bye!


## <span style='color:#7e22ce'>Example 3: Chat with any PDF</span>

The whole pipeline in **one cell** that you can copy into your own projects. Change `PDF` to the path of any PDF and run it. The default is an M.Phil transcript (personal details removed). This cell imports everything it needs, so it also works on its own.

In [12]:
# ---------------- Chat with any PDF: load -> split -> embed + store -> hybrid search + rerank -> answer ----------------
import os, re
from dotenv import load_dotenv
from transformers import AutoTokenizer
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

PDF = "../data/arif_mphil_transcript_redacted.pdf"                 # <- change this to any PDF

load_dotenv("../keys/.env", override=True)
llm = ChatOpenAI(model="openai/gpt-oss-120b", base_url="https://api.groq.com/openai/v1",
                 api_key=os.getenv("GROQ_API_KEY"), max_retries=5)
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# 1-3. index: load, split to 200 tokens, embed and store
pdf_chunks = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tok, chunk_size=200, chunk_overlap=20) \
                 .split_documents(PyPDFLoader(PDF).load())
store = Chroma.from_documents(pdf_chunks, emb, collection_name="any_pdf", collection_metadata={"hnsw:space": "cosine"})

# 4. retrieve: BM25 + vectors, then keep the best 4 with a cross-encoder
pdf_retriever = ContextualCompressionRetriever(
    base_retriever=EnsembleRetriever(retrievers=[
        BM25Retriever.from_documents(pdf_chunks, k=10, preprocess_func=lambda t: re.findall(r"\w+", t.lower())),
        store.as_retriever(search_kwargs={"k": 10})]),
    base_compressor=CrossEncoderReranker(model=HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L6-v2"), top_n=4))

# 5-6. augment and generate
pdf_prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about a document. Use ONLY the context below. Cite pages like [p.1]. "
               "If the answer is not in the context, say: I don't know."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")])
pdf_chain = (
    RunnableParallel(docs=pdf_retriever, question=RunnablePassthrough())
    | RunnablePassthrough.assign(context=lambda x: "\n\n".join(f"[p.{d.metadata['page_label']}] {d.page_content}" for d in x["docs"]))
    | RunnablePassthrough.assign(answer=pdf_prompt | llm | StrOutputParser()))

print(f"{PDF}: {len(pdf_chunks)} chunks indexed. Ask your questions (Enter to stop).\n")
while True:
    question = input("You: ")
    if not question.strip():
        break
    print("Bot:", pdf_chain.invoke(question)["answer"], "\n")

../data/arif_mphil_transcript_redacted.pdf: 2 chunks indexed. Ask your questions (Enter to stop).

You: What grade did I get in CS-565?
Bot: You received an **A** grade in CS‑565. [ p.1 ] 

You: What was the final CGPA?
Bot: The transcript lists the final cumulative GPA as **3.85**【p.1】. 

You: What was the title of the thesis?
Bot: The thesis was titled **“Taxonomy of IDS: Intrusion Detection using System Call Sequence.”**【p.1】 

You: Which course had the highest marks?
Bot: The highest marks recorded are **90**, achieved in two courses:

- **CS‑520 – Advance Computer Architecture** (90)  
- **CS‑500 – Digital Signal Processing** (90)

Both of these courses received the top score of 90 % on the transcript [p.1]. 

You: 


> **Note:** this transcript is only one page, so it fits in 2 chunks and every question sees all of it. With a long PDF, the same code finds the best 4 chunks out of hundreds. Try `PDF = "../data/attention.pdf"`.

# <span style='background :lightgreen' >7. Vectorless RAG</span>

## <span style='color:#7e22ce'>Example 4: An LLM reads the table of contents instead of searching vectors</span>

Vector search finds text that is **similar** to the question. A person looking for "the hardware used for training" in a paper does something else: they open the **table of contents**, see the section *5.2 Hardware and Schedule*, and read that section. **Vectorless (reasoning-based) RAG** lets the LLM do the same:
1. **Index = a tree:** the document's sections, with their titles and pages. No chunking, no embeddings, no vector store.
2. **Search = reasoning:** the LLM reads the table of contents and **chooses** the sections that should contain the answer.
3. **Answer:** the LLM reads the **full** chosen sections and answers, citing section and page.

This is the idea of [PageIndex](https://pageindex.ai/blog/pageindex-intro) (2025), also described by [Microsoft (Mar 2026)](https://techcommunity.microsoft.com/blog/azuredevcommunityblog/vectorless-reasoning-based-rag-a-new-approach-to-retrieval-augmented-generation/4502238). PageIndex builds the tree with an LLM and also writes a summary of every section; we build a simple version from the paper's numbered headings.

### Step 1: Build the tree from the headings

In [13]:
heading = re.compile(r"^(\d+(?:\.\d+)*)\s+([A-Z][A-Za-z\-\s,()]{2,60})$")      # e.g. "5.2 Hardware and Schedule"

sections = [{"id": "0", "title": "Title, authors and abstract", "page": "1", "text": ""}]   # the text before section 1
current = sections[0]
for page in pages:                                                # pages from Example 1, Step 1
    for line in page.page_content.splitlines():
        s = line.strip()
        match = heading.match(s)
        if (match and not re.search(r"\d{3,}", s)) or s == "References":          # a new section starts here
            current = {"id": match.group(1) if match else "R",
                       "title": match.group(2).strip() if match else "References",
                       "page": page.metadata["page_label"], "text": ""}
            sections.append(current)
        else:
            current["text"] += line + "\n"                        # every other line belongs to the current section

by_id = {s["id"]: s for s in sections}                            # find a section by its id
toc = "\n".join(f'[{s["id"]}] {s["title"]} (page {s["page"]}): {" ".join(s["text"].split())[:100]}...'
                for s in sections)                                # the table of contents the LLM will read
print(len(sections), "sections; the table of contents has", len(toc), "characters\n")
print("\n".join(f'[{s["id"]:>5}] {s["title"]} (p.{s["page"]})' for s in sections))

24 sections; the table of contents has 3201 characters

[    0] Title, authors and abstract (p.1)
[    1] Introduction (p.2)
[    2] Background (p.2)
[    3] Model Architecture (p.2)
[  3.1] Encoder and Decoder Stacks (p.3)
[  3.2] Attention (p.3)
[3.2.1] Scaled Dot-Product Attention (p.4)
[3.2.2] Multi-Head Attention (p.4)
[3.2.3] Applications of Attention in our Model (p.5)
[  3.3] Position-wise Feed-Forward Networks (p.5)
[  3.4] Embeddings and Softmax (p.5)
[  3.5] Positional Encoding (p.6)
[    4] Why Self-Attention (p.6)
[    5] Training (p.7)
[  5.1] Training Data and Batching (p.7)
[  5.2] Hardware and Schedule (p.7)
[  5.3] Optimizer (p.7)
[  5.4] Regularization (p.7)
[    6] Results (p.8)
[  6.1] Machine Translation (p.8)
[  6.2] Model Variations (p.8)
[  6.3] English Constituency Parsing (p.9)
[    7] Conclusion (p.10)
[    R] References (p.10)


### Step 2: The LLM chooses the sections
We ask for **structured output** (Lec-30): the ids of the sections to read, and one sentence of reasoning, which makes the choice easy to check.

In [14]:
from pydantic import BaseModel, Field

class SectionChoice(BaseModel):
    reasoning: str = Field(description="one sentence: why these sections")
    section_ids: list[str] = Field(description="ids of the 1 to 3 sections most likely to contain the answer")

chooser = GROQ_MODEL.with_structured_output(SectionChoice)       # a model that returns a SectionChoice object

question = "What hardware was used to train the models, and for how long?"
choice = chooser.invoke(
    "You are navigating a research paper using its table of contents.\n"
    f"TABLE OF CONTENTS:\n{toc}\n\nQUESTION: {question}\nWhich sections should be read to answer it?")
print("sections:", choice.section_ids)
print("why     :", choice.reasoning)

sections: ['5.2']
why     : The hardware details and training duration are described in the Hardware and Schedule subsection of the Training section.


### Step 3: Read the full sections and answer

In [15]:
section_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY these sections of a paper. Cite them like [5.2, p.7]. "
               "If the answer is not there, say: I don't know."),
    ("human", "{sections}\n\nQuestion: {question}"),
])

def read_sections(ids) -> str:
    """The full text of the chosen sections, each with its id, title and page."""
    return "\n\n".join(f'[{i}, p.{by_id[i]["page"]}] {by_id[i]["title"]}\n{by_id[i]["text"]}' for i in ids if i in by_id)

answer = (section_prompt | GROQ_MODEL | StrOutputParser()).invoke(
    {"sections": read_sections(choice.section_ids), "question": question})
print(answer)

The models were trained on a single machine equipped with **8 NVIDIA P100 GPUs**.  
- The base models were trained for **100,000 steps**, which took about **12 hours**.  
- The larger models were trained for **300,000 steps**, requiring roughly **3.5 days**.  

[5.2, p.7]


### Vector RAG and vectorless RAG on the same questions
We put Steps 2 and 3 into one function and ask both pipelines the same questions:

In [16]:
def vectorless_rag(question: str) -> dict:
    """Choose sections from the table of contents, then answer from their full text."""
    choice = chooser.invoke("You are navigating a research paper using its table of contents.\n"
                            f"TABLE OF CONTENTS:\n{toc}\n\nQUESTION: {question}\nWhich sections should be read to answer it?")
    answer = (section_prompt | GROQ_MODEL | StrOutputParser()).invoke(
        {"sections": read_sections(choice.section_ids), "question": question})
    return {"sections": choice.section_ids, "answer": answer}

for q in ["Who are the authors of the paper?",
          "What BLEU score did the big Transformer get on English-to-German?",
          "What happens to translation quality with a single attention head?"]:
    vec = rag_chain.invoke(q)                                    # Example 1: hybrid search + reranking
    tree = vectorless_rag(q)                                     # Example 4: table of contents + reasoning
    print("Q:", q)
    print("   vector RAG     reads chunks from pages", sorted({d.metadata["page_label"] for d in vec["docs"]}, key=int))
    print("     ->", vec["answer"][:160].replace("\n", " "))
    print("   vectorless RAG reads sections", tree["sections"])
    print("     ->", tree["answer"][:160].replace("\n", " "), "\n")

Q: Who are the authors of the paper?
   vector RAG     reads chunks from pages ['1', '11', '12']
     -> The paper was authored by Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin [p.1].
   vectorless RAG reads sections ['0']
     -> The authors of the paper are Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin [0, p. 

Q: What BLEU score did the big Transformer get on English-to-German?
   vector RAG     reads chunks from pages ['1', '8', '9']
     -> The big Transformer achieved a BLEU score of **28.4** on the WMT 2014 English‑to‑German translation task【p.8】【p.1】.
   vectorless RAG reads sections ['6.1', '6', '0']
     -> The big Transformer achieved a BLEU score of **28.4** on the WMT 2014 English‑to‑German translation task [6.1, p.8]. 

Q: What happens to translation quality with a single attention head?
   vector RAG     reads chunk

Both pipelines answer these questions. They get there differently:

<div style="text-align:center;">
    <img src="../images/langchain-new-35.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

| Feature | Vector RAG | Vectorless (reasoning-based) RAG |
|---|---|---|
| **Index** | chunks + embeddings in a vector store | a tree of sections (titles, pages, optional summaries) |
| **How it searches** | similarity of vectors (plus BM25 in hybrid search) | an LLM **reasons** over the table of contents |
| **Unit given to the model** | chunks of a few hundred tokens | whole sections |
| **Embedding model / vector DB** | needed | not needed |
| **LLM calls per question** | 1 (answer) | 2 or more (choose, then answer) |
| **Explaining a result** | "these chunks were similar" | "I chose section 5.2 because ..." |
| **Strong for** | many documents, unstructured text, chat, product search | long structured documents: reports, contracts, manuals, papers |
| **Weak for** | split tables and sections, exact references ("Table 3", "step 3") | documents without structure, very large collections, low latency |
| **Try it** | [notebooklm.google.com](https://notebooklm.google.com) | [chat.pageindex.ai](https://chat.pageindex.ai) |

> **Note:** PageIndex reports 98.7% accuracy on FinanceBench (questions about financial reports) for its own system; that is a vendor result, not an independent benchmark. Keyword search (BM25, Lec-37) is sometimes also called "vectorless", but it is a different idea: it matches words, while reasoning-based RAG lets the LLM navigate the document.

### Try it yourself: the same PDF in both tools
Upload the handout [HO 1.4 (Recap of Internetworking Concepts with Linux)](https://www.arifbutt.me/wp-content/uploads/2025/04/Handout-1.4-Recap-of-InterNetworking-Concepts-with-Linux.pdf) to NotebookLM (vector RAG) and to PageIndex (vectorless RAG), and compare the answers to:
- "List the names of the attacks that can be launched on the Internet layer."
- "What port number does the web server listen on, and what is the exact subnet mask used in the VM networking example?" (exact numbers)
- "What is the exact Linux command used to configure the BIND DNS server?" (exact commands)
- "List all TCP/IP stack layers mentioned in the document along with their corresponding protocols." (a table)
- "How does the firewall configuration relate to the client-server services described? Which ports are explicitly allowed?" (two distant sections)
- "What is the third step in setting up the virtual machine network interface?" (order of steps)
- "What does the network topology diagram in the document illustrate about VM interconnections?" (a figure)

# <span style='background :lightgreen' >8. Summary</span>

| Ex | What | Pipeline |
|---|---|---|
| 1 | A research paper, step by step | load → split → embed + store → hybrid search + rerank → prompt → model, then one LCEL chain |
| 2 | A YouTube lecture | transcript with times → the same pipeline → a question loop |
| 3 | Any PDF | the whole pipeline in one cell, with a question loop |
| 4 | Vectorless RAG | table of contents → the LLM chooses sections → answer from full sections |

### Key takeaways
- **RAG** = retrieve the relevant passages, add them to the prompt, generate the answer from them.
- **Indexing** happens once (load, split, embed, store); **querying** for every question (retrieve, augment, generate).
- Naive RAG breaks in predictable places; **advanced RAG** fixes them with better chunks, hybrid search, reranking and grounded prompts with citations.
- Newer RAG types change the **control flow** (Self-RAG, Corrective, Adaptive, Speculative, Agentic) or the **knowledge structure** (GraphRAG, vectorless).
- **Vectorless RAG** replaces similarity search with an LLM reasoning over the document's structure: no embeddings, but more LLM calls.
- **Measure** with test questions before adding complexity.
- **Next (Lec-39):** from RAG to **agents**: an LLM that decides itself when to search, which tool to use, and when it is done.

### Check your understanding
1. What do the three letters of RAG stand for, and which LangChain component does each step use?
2. Which steps happen once, and which happen for every question?
3. Name three of the seven failure points of naive RAG and a fix for each.
4. How does Corrective RAG differ from Self-RAG?
5. What does Adaptive-RAG decide, and why does that save cost?
6. When would GraphRAG answer better than vector RAG?
7. In Example 1, why did hybrid search with reranking find the authors when plain vector search did not?
8. Why did the model answer "I don't know" for the capital of France, although the retriever returned 4 chunks?
9. In Example 2, why do we keep the start time of every chunk?
10. How does vectorless RAG find the right section without any embeddings? What are its costs and limits?